# PBMC model comparison

Audience:
- Researchers who want a quick baseline benchmark before building a custom model.

Prerequisites:
- Install `scdlkit[tutorials]`.
- Know the Scanpy PBMC quickstart notebook first.

Learning goals:
- Compare `autoencoder`, `vae`, and `transformer_ae` on the same PBMC workflow.
- Inspect the metrics table and saved comparison plot.
- Understand which baseline is strongest for a quick sanity check.


## Outline

1. Load PBMC data with Scanpy.
2. Detect the runtime device.
3. Run `compare_models` with shared settings.
4. Review scalar metrics.
5. Inspect the generated benchmark plot and saved outputs.


In [ ]:
from __future__ import annotations

from pathlib import Path

import scanpy as sc
import torch
from IPython.display import Image, display

from scdlkit import compare_models

DATA_PATH = Path("examples/data/pbmc3k_processed.h5ad")
OUTPUT_DIR = Path("artifacts/pbmc_compare")

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_name}")


In [ ]:
adata = sc.read_h5ad(DATA_PATH) if DATA_PATH.exists() else sc.datasets.pbmc3k_processed()
print(adata)
print("Label field used for comparison:", "louvain")


## Compare baseline models

The benchmark uses the same task, label field, and training settings so the model family is the main variable.


In [ ]:
result = compare_models(
    adata,
    models=["autoencoder", "vae", "transformer_ae"],
    task="representation",
    shared_kwargs={
        "epochs": 5,
        "batch_size": 128,
        "label_key": "louvain",
        "device": "auto",
    },
    output_dir=str(OUTPUT_DIR),
)
result.metrics_frame


## Generated outputs

The comparison helper writes a CSV summary, a Markdown report, and a PNG plot to `artifacts/pbmc_compare/`.


In [ ]:
result.output_paths


In [ ]:
if "comparison_png" in result.output_paths:
    display(Image(filename=result.output_paths["comparison_png"]))


## Short interpretation

A useful baseline workflow is to start with the strongest simple model here, then use that as the reference point for any custom research architecture.


In [ ]:
scalar_columns = [col for col in result.metrics_frame.columns if col != "model"]
summary_metric = "silhouette" if "silhouette" in scalar_columns else scalar_columns[0]
best_row = result.metrics_frame.sort_values(summary_metric, ascending=False).iloc[0]
print(f"Best model by {summary_metric}: {best_row['model']}")
